In [ ]:
import seaborn as sns
import pandas as pd
df = sns.load_dataset("titanic")
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [ ]:
before_rows = len(df)
before_nulls = int(df.isnull().sum().sum())
before_dups = int(df.duplicated().sum())
print("Shape:", df.shape)
print("Nulls per column:")
print(df.isnull().sum())
print("Duplicate rows:", before_dups)
print("Data types:")
print(df.dtypes)
print(df.describe())

Shape: (891, 15)
Nulls per column:
survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64
Duplicate rows: 107
Data types:
survived          int64
pclass            int64
sex              object
age             float64
sibsp             int64
parch             int64
fare            float64
embarked         object
class          category
who              object
adult_male         bool
deck           category
embark_town      object
alive            object
alone              bool
dtype: object
         survived      pclass         age       sibsp       parch        fare
count  891.000000  891.000000  714.000000  891.000000  891.000000  891.000000
mean     0.383838    2.308642   29.699118    0.523008    0.381594   32.204208
std      0.486592    0.8360

In [ ]:
df = df.drop_duplicates()
print("Duplicates removed:", before_dups)
df["age"] = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df["embark_town"] = df["embark_town"].fillna(df["embark_town"].mode()[0])
df = df.drop(columns=["deck"])
print("Nulls left:", int(df.isnull().sum().sum()))
print("Shape now:", df.shape)

Duplicates removed: 107
Nulls left: 0
Shape now: (784, 14)


/tmp/ipykernel_2501/3909732706.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["age"] = df["age"].fillna(df["age"].median())
/tmp/ipykernel_2501/3909732706.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
/tmp/ipykernel_2501/3909732706.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://

## Missing Data Handling
- **age**: filled with the median, because it is numeric and the median is not affected by extreme values.
- **embarked / embark_town**: filled with the mode (most common value), because they are categorical and only a few values were missing.
- **deck**: column dropped, because about 77% of its values were missing and filling it would add fake information.
- **Duplicates**: duplicate rows were removed.

In [ ]:
df["sex"] = df["sex"].str.capitalize()
df["age"] = df["age"].round().astype(int)
df["survived"] = df["survived"].astype(bool)
df["pclass"] = df["pclass"].astype("category")
q1 = df["fare"].quantile(0.25)
q3 = df["fare"].quantile(0.75)
iqr = q3 - q1
low = q1 - 1.5 * iqr
high = q3 + 1.5 * iqr
print("Fare outliers:", int(((df["fare"] < low) | (df["fare"] > high)).sum()))
df["fare"] = df["fare"].clip(lower=low, upper=high)
print(df.dtypes)

Fare outliers: 97
survived           bool
pclass         category
sex              object
age               int64
sibsp             int64
parch             int64
fare            float64
embarked         object
class          category
who              object
adult_male         bool
embark_town      object
alive            object
alone              bool
dtype: object


In [ ]:
df = df.drop_duplicates()
after_rows = len(df)
after_nulls = int(df.isnull().sum().sum())
after_dups = int(df.duplicated().sum())
summary = pd.DataFrame([["Rows", before_rows, after_rows], ["Null values", before_nulls, after_nulls], ["Duplicate rows", before_dups, after_dups]], columns=["Metric", "Before", "After"])
print(summary)
df.to_csv("cleaned_titanic.csv", index=False)
from google.colab import files
files.download("cleaned_titanic.csv")

           Metric  Before  After
0            Rows     891    765
1     Null values     869      0
2  Duplicate rows     107      0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Standardisation, Data Types and Outliers
- **sex**: converted to a consistent format (Male / Female).
- **age**: converted from decimal to whole numbers.
- **survived**: converted to True/False. **pclass**: converted to a category.
- **fare outliers**: 97 outliers detected using the IQR method and capped (not removed), because high fares are real first-class tickets and removing them would lose real information.

## Before vs After
- **Data types before:** age was float, survived was int, pclass was int.
- **Data types after:** age is int, survived is bool, pclass is category.

## Final Duplicate Check
After filling missing values and standardising data types, 15 duplicate rows appeared. A second duplicate removal brought the final duplicate count to 0.